# IMDb archived ratings pages via Tor

For every IMDb ID in the input CSV, this notebook lists Wayback Machine captures of `/title/{imdb_id}/ratings` from 2005 through 2025, selects the latest capture in each year, and tries older captures from that same year if playback fails. Successful HTML pages are saved under `ratings` as `{14-digit-timestamp}_{imdb_id}.html`.

Run the code cells from top to bottom. This reuses the CDX, Tor SOCKS proxy, and Selenium playback workflow from `imdb_archive_scraper.ipynb`. The download cell uses `MAX_PARALLEL_SCRAPES` to set the number of concurrent movie/year downloads (and Selenium sessions); start low because all sessions share the same Tor proxy.

Dependencies: `python -m pip install "requests[socks]" selenium` (plus Chrome/ChromeDriver available to Selenium). Tor must be running on `127.0.0.1:9050`.

In [1]:
import csv
import json
import random
import re
import socket
import time
from pathlib import Path
from urllib.parse import urlencode

import requests
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

CSV_PATH = Path("imdb_ids_with_titles.csv")
LIMIT = 0  # 0 means all movies; set a positive number for a small test run.
START_YEAR = 2005
END_YEAR = 2025
YEARS = range(START_YEAR, END_YEAR + 1)
CDX_URL = "https://web.archive.org/cdx/search/cdx"
RESPONSES_DIR = Path("responses_ratings")
RATINGS_DIR = Path("ratings")
TOR_HOST, TOR_PORT = "127.0.0.1", 9050
PROXIES = {
    "http": f"socks5h://{TOR_HOST}:{TOR_PORT}",
    "https": f"socks5h://{TOR_HOST}:{TOR_PORT}",
}

RESPONSES_DIR.mkdir(parents=True, exist_ok=True)
RATINGS_DIR.mkdir(parents=True, exist_ok=True)

try:
    with socket.create_connection((TOR_HOST, TOR_PORT), timeout=5):
        print("proxy port is reachable")
except OSError as error:
    print(f"proxy connection failed: {error}")

proxy port is reachable


In [2]:
def movie_ids(csv_path):
    with csv_path.open(newline="", encoding="utf-8-sig") as source:
        reader = csv.DictReader(source)
        if "imdb_id" not in (reader.fieldnames or []):
            raise ValueError(f"{csv_path} must contain an 'imdb_id' column")

        seen = set()
        for row in reader:
            imdb_id = (row.get("imdb_id") or "").strip()
            if not re.fullmatch(r"tt\d+", imdb_id):
                raise ValueError(f"Invalid IMDb ID in CSV: {imdb_id!r}")
            if imdb_id not in seen:
                seen.add(imdb_id)
                yield imdb_id


def fetch_captures(imdb_id, responses_dir):
    params = urlencode({
        "url": f"https://www.imdb.com/title/{imdb_id}/ratings",
        "output": "json",
        "fl": "timestamp,original,statuscode,mimetype,digest",
        "filter": ["statuscode:200", "mimetype:text/html"],
    }, doseq=True)
    response = requests.get(f"{CDX_URL}?{params}", proxies=PROXIES, timeout=60)
    response.raise_for_status()

    cdx_data = response.content
    (responses_dir / f"response_{imdb_id}.txt").write_bytes(cdx_data)

    captures = json.loads(cdx_data)
    if len(captures) < 2:
        return []
    return captures[1:]  # Drop the header row.

In [3]:
def getPage(url, folder, exact_date, page_name, group_name, file_extension):
    output_dir = Path(folder)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / f"{exact_date}_{page_name}.{file_extension}"

    if output_file.exists():
        print(f"\nSKIP {exact_date}: Already downloaded")
        return True

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument(f"--proxy-server=socks5://{TOR_HOST}:{TOR_PORT}")
    driver = webdriver.Chrome(options=options)

    try:
        driver.set_page_load_timeout(90)

        for attempt in range(1):
            try:
                driver.get("https://check.torproject.org/")
                if (
                    driver.title == "Congratulations. This browser is configured to use Tor."
                    and driver.current_url == "https://check.torproject.org/"
                ):
                    print("__TOR is OK")

                driver.get(url)
                time.sleep(3)
                html = driver.page_source

                if any(message in html for message in (
                    "403 Forbidden",
                    "Error 403",
                    "ERR_TOO_MANY_REDIRECTS",
                    "redirected you too many times",
                    "This site can’t be reached",
                )):
                    print(f"NOT SCRAPPABLE: {exact_date}")
                    return False

                if "429 Too Many Requests" in html:
                    print(f"429 RATE LIMIT: {exact_date}")
                    return None

                if len(html) < 1000:
                    raise ValueError("Page too short")

                output_file.write_text(html, encoding="utf-8")
                print(f"OK {exact_date}: {output_file}")
                return True

            except Exception as error:
                print(f"Attempt failed: {exact_date}: chromedriver error\n")
                if attempt < 2:
                    time.sleep(10)

        return False
    finally:
        driver.quit()

In [4]:
movie_id_list = list(movie_ids(CSV_PATH))
selected_movie_ids = movie_id_list if LIMIT == 0 else movie_id_list[:LIMIT]

# captures_by_movie = {}
# available_capture_dates = []
# capture_errors = {}

# for position, imdb_id in enumerate(selected_movie_ids, start=1):
#     print(f"{position}/{len(selected_movie_ids)}: fetching capture dates for {imdb_id}")
#     try:
#         captures = fetch_captures(imdb_id, RESPONSES_DIR)
#         captures_by_movie[imdb_id] = []
#         for capture in captures:
#             timestamp, original = capture[:2]
#             if len(timestamp) < 4 or not timestamp[:4].isdigit():
#                 continue
#             year = int(timestamp[:4])
#             if START_YEAR <= year <= END_YEAR:
#                 captures_by_movie[imdb_id].append(capture)
#                 available_capture_dates.append({
#                     "imdb_id": imdb_id,
#                     "year": year,
#                     "timestamp": timestamp,
#                     "date": timestamp[:8],
#                     "original": original,
#                 })
#     except Exception as error:
#         captures_by_movie[imdb_id] = []
#         capture_errors[imdb_id] = str(error)
#         print(f"ERROR fetching captures for {imdb_id}: {error}")

# print(f"\nAvailable capture records from {START_YEAR} through {END_YEAR}: {len(available_capture_dates)}")
# for capture in available_capture_dates:
#     print(
#         f"{capture['imdb_id']} | {capture['year']} | "
#         f"{capture['date']} ({capture['timestamp']}) | {capture['original']}"
#     )

In [5]:
import csv
from pathlib import Path

USE_SAVED_CAPTURE_DATES = True
CAPTURE_DATES_CSV = Path("ratings_capture_dates.csv")

captures_by_movie = {imdb_id: [] for imdb_id in selected_movie_ids}
available_capture_dates = []
capture_errors = {}

if USE_SAVED_CAPTURE_DATES:
    with CAPTURE_DATES_CSV.open(newline="", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file)
        required = {"imdb_id", "year", "timestamp", "date", "original"}
        if not required.issubset(reader.fieldnames or []):
            raise ValueError(f"{CAPTURE_DATES_CSV} must contain columns: {sorted(required)}")

        selected_ids = set(selected_movie_ids)
        for row in reader:
            imdb_id = (row["imdb_id"] or "").strip()
            if imdb_id not in selected_ids:
                continue

            timestamp = (row["timestamp"] or "").strip()
            original = (row["original"] or "").strip()
            if not timestamp or not original:
                raise ValueError(f"Missing timestamp or original URL for {imdb_id}")

            capture = {
                "imdb_id": imdb_id,
                "year": int(row["year"]),
                "timestamp": timestamp,
                "date": row["date"],
                "original": original,
            }
            available_capture_dates.append(capture)

            # The later scrape cells use only capture[0] and capture[1].
            captures_by_movie[imdb_id].append([timestamp, original])

    print(f"Loaded {len(available_capture_dates)} capture records from {CAPTURE_DATES_CSV}")

else:
    for position, imdb_id in enumerate(selected_movie_ids, start=1):
        print(f"{position}/{len(selected_movie_ids)}: fetching capture dates for {imdb_id}")
        try:
            captures = fetch_captures(imdb_id, RESPONSES_DIR)
            for timestamp, original, *_ in captures:
                if len(timestamp) < 4 or not timestamp[:4].isdigit():
                    continue

                year = int(timestamp[:4])
                if START_YEAR <= year <= END_YEAR:
                    captures_by_movie[imdb_id].append([timestamp, original])
                    available_capture_dates.append({
                        "imdb_id": imdb_id,
                        "year": year,
                        "timestamp": timestamp,
                        "date": timestamp[:8],
                        "original": original,
                    })
        except Exception as error:
            capture_errors[imdb_id] = str(error)
            print(f"ERROR fetching captures for {imdb_id}")

print(f"Available capture records: {len(available_capture_dates)}")

Loaded 9005 capture records from ratings_capture_dates.csv
Available capture records: 9005


In [6]:
captures_by_movie_year = {
    imdb_id: {year: [] for year in YEARS}
    for imdb_id in selected_movie_ids
}

for imdb_id, captures in captures_by_movie.items():
    for capture in captures:
        year = int(capture[0][:4])
        captures_by_movie_year[imdb_id][year].append(capture)

latest_capture_for_year = {}
for imdb_id, year_captures in captures_by_movie_year.items():
    for year, captures in year_captures.items():
        captures.sort(key=lambda capture: capture[0], reverse=True)
        if captures:
            latest_capture_for_year[(imdb_id, year)] = captures[0]

print(f"Latest capture selected for {len(latest_capture_for_year)} movie/year pairs:")
for (imdb_id, year), capture in latest_capture_for_year.items():
    print(f"{imdb_id} | {year} | {capture[0]} | {capture[1]}")

Latest capture selected for 5153 movie/year pairs:
tt0068646 | 2005 | 20051222185933 | http://www.imdb.com:80/title/tt0068646/ratings
tt0068646 | 2006 | 20061225093810 | http://www.imdb.com:80/title/tt0068646/ratings
tt0068646 | 2007 | 20071012081216 | http://www.imdb.com:80/title/tt0068646/ratings
tt0068646 | 2008 | 20081224162429 | http://www.imdb.com:80/title/tt0068646/ratings
tt0068646 | 2010 | 20100723204921 | http://www.imdb.com:80/title/tt0068646/ratings
tt0068646 | 2011 | 20111203214435 | http://www.imdb.com:80/title/tt0068646/ratings
tt0068646 | 2012 | 20120725090729 | http://www.imdb.com/title/tt0068646/ratings
tt0068646 | 2013 | 20130429171146 | http://www.imdb.com/title/tt0068646/ratings
tt0068646 | 2014 | 20140714192629 | http://www.imdb.com/title/tt0068646/ratings
tt0068646 | 2015 | 20150822234408 | http://www.imdb.com:80/title/tt0068646/ratings
tt0068646 | 2016 | 20161206184656 | http://www.imdb.com:80/title/tt0068646/ratings?
tt0068646 | 2017 | 20171129183644 | http://w

In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_PARALLEL_SCRAPES = 8  # Tune this integer to control concurrent Selenium sessions.
if MAX_PARALLEL_SCRAPES < 1:
    raise ValueError("MAX_PARALLEL_SCRAPES must be at least 1")


def scrape_movie_year(imdb_id, year):
    candidates = captures_by_movie_year[imdb_id][year]
    if not candidates:
        return {
            "imdb_id": imdb_id,
            "year": year,
            "status": "no_capture" if imdb_id not in capture_errors else "capture_error",
            "timestamp": "",
            "attempts": 0,
            "error": capture_errors.get(imdb_id, ""),
        }

    last_error = ""
    for attempt_number, capture in enumerate(candidates, start=1):
        timestamp, original = capture[:2]
        archive_url = f"https://web.archive.org/web/{timestamp}id_/{original}"
        try:
            succeeded = getPage(
                archive_url,
                RATINGS_DIR,
                timestamp,
                imdb_id,
                imdb_id,
                "html",
            ) is True
        except Exception as error:
            last_error = str(error)
            print(f"ERROR {imdb_id} {year} {timestamp}")
            succeeded = False

        if succeeded:
            return {
                "imdb_id": imdb_id,
                "year": year,
                "status": "downloaded",
                "timestamp": timestamp,
                "attempts": attempt_number,
                "error": "",
            }

        if attempt_number < len(candidates):
            time.sleep(random.uniform(1, 4))

    print(f"FAILED {imdb_id} {year}: tried {len(candidates)} captures from {year}")
    return {
        "imdb_id": imdb_id,
        "year": year,
        "status": "failed",
        "timestamp": "",
        "attempts": len(candidates),
        "error": last_error,
    }


scrape_tasks = [
    (movie_position, imdb_id, year)
    for movie_position, imdb_id in enumerate(selected_movie_ids, start=1)
    for year in YEARS
]
results_by_task = {}

with ThreadPoolExecutor(max_workers=MAX_PARALLEL_SCRAPES) as executor:
    future_to_task = {
        executor.submit(scrape_movie_year, imdb_id, year): (movie_position, imdb_id, year)
        for movie_position, imdb_id, year in scrape_tasks
    }

    for completed, future in enumerate(as_completed(future_to_task), start=1):
        task = future_to_task[future]
        results_by_task[task] = future.result()
        if completed % 25 == 0 or completed == len(future_to_task):
            print(f"Completed {completed}/{len(future_to_task)} movie/year tasks")

download_results = [results_by_task[task] for task in scrape_tasks]
successful_downloads = sum(result["status"] == "downloaded" for result in download_results)
print(
    f"\nFinished: {successful_downloads} downloaded; "
    f"{len(download_results) - successful_downloads} movie/year pairs without a downloaded page."
)
for status in ("downloaded", "failed", "no_capture", "capture_error"):
    print(f"{status}: {sum(result['status'] == status for result in download_results)}")
print(f"HTML files are in: {RATINGS_DIR.resolve()}")
if capture_errors:
    print(f"Capture-list errors for {len(capture_errors)} movies; see capture_errors.")


SKIP 20051222185933: Already downloaded
SKIP 20061225093810: Already downloaded


SKIP 20071012081216: Already downloaded

SKIP 20081224162429: Already downloaded

SKIP 20100723204921: Already downloaded

SKIP 20111203214435: Already downloaded

SKIP 20120725090729: Already downloaded

SKIP 20140714192629: Already downloaded

SKIP 20150822234408: Already downloaded

SKIP 20130429171146: Already downloaded

SKIP 20161206184656: Already downloaded

SKIP 20171129183644: Already downloaded

SKIP 20180801112134: Already downloaded

SKIP 20210224163451: Already downloaded

SKIP 20221010144411: Already downloaded

SKIP 20231024212241: Already downloaded

SKIP 20051219143841: Already downloaded

SKIP 20060829221735: Already downloaded

SKIP 20071016122744: Already downloaded

SKIP 20081228062507: Already downloaded

SKIP 20101006053437: Already downloaded

SKIP 20111214220415: Already downloaded

SKIP 20121112194904: Already downloaded

SKIP 20130430195341: Already downloaded

SKIP 2020112902

Metadata file /Users/surenpoghosyan/.cache/selenium/se-metadata.json cannot be parsed: EOF while parsing a value at line 25 column 22. Using empty metadata
Metadata file /Users/surenpoghosyan/.cache/selenium/se-metadata.json cannot be parsed: EOF while parsing a string at line 25 column 36. Using empty metadata


OK 20170308212819: ratings/20170308212819_tt0118799.htmlAttempt failed: 20170308212819: chromedriver error

FAILED tt0118799 2020: tried 1 captures from 2020

SKIP 20061218113427: Already downloaded

SKIP 20101002035032: Already downloaded

SKIP 20111214022835: Already downloaded

SKIP 20120422222757: Already downloaded

SKIP 20130425220930: Already downloaded

SKIP 20140714203422: Already downloaded

SKIP 20151220181629: Already downloaded

SKIP 20161227064720: Already downloaded

SKIP 20181018215845: Already downloaded

SKIP 20190506084308: Already downloaded

SKIP 20221226184037: Already downloaded

SKIP 20241002034953: Already downloaded

SKIP 20051031145327: Already downloaded

SKIP 20070302070843: Already downloaded

SKIP 20080416190722: Already downloaded

SKIP 20090321050015: Already downloaded

SKIP 20110302202501: Already downloaded

SKIP 20120316034427: Already downloaded

SKIP 20131218073609: Already downloaded
Completed 525/12789 movie/year tasks

SKIP 20140226204746: Alre

Metadata file /Users/surenpoghosyan/.cache/selenium/se-metadata.json cannot be parsed: EOF while parsing a string at line 18 column 8. Using empty metadata


Attempt failed: 20110521234111: chromedriver error
Attempt failed: 20180312041637: chromedriver error

Attempt failed: 20140215192206: chromedriver error

Attempt failed: 20120423174337: chromedriver error

ERROR tt0089881 2011 20110521234111
FAILED tt0089881 2011: tried 1 captures from 2011
Attempt failed: 20130908205956: chromedriver error

__TOR is OK
__TOR is OK
__TOR is OK
__TOR is OK
__TOR is OK
__TOR is OK
__TOR is OK
__TOR is OK
Attempt failed: 20121213065329: chromedriver error

OK 20061101090014: ratings/20061101090014_tt0089881.html
Attempt failed: 20250620154821: chromedriver error

__TOR is OK
OK 20220901051134: ratings/20220901051134_tt0118799.html
FAILED tt0338013 2018: tried 1 captures from 2018

SKIP 20190820172137: Already downloaded

SKIP 20200906163234: Already downloaded

SKIP 20210125023709: Already downloaded

SKIP 20221118211245: Already downloaded

SKIP 20250612022353: Already downloaded

SKIP 20050401014152: Already downloaded

SKIP 20060827171930: Already dow

Metadata file /Users/surenpoghosyan/.cache/selenium/se-metadata.json cannot be parsed: EOF while parsing a string at line 5 column 10. Using empty metadata


... (393 outputs left out)

Metadata file /Users/surenpoghosyan/.cache/selenium/se-metadata.json cannot be parsed: EOF while parsing a string at line 6 column 29. Using empty metadata


FAILED tt10431500 2023: tried 1 captures from 2023OK 20130916030745: ratings/20130916030745_tt0097223.html
Completed 12100/12789 movie/year tasks
FAILED tt0097223 2021: tried 1 captures from 2021
OK 20220930090421: ratings/20220930090421_tt0097223.html
__TOR is OK
OK 20220925134754: ratings/20220925134754_tt2948372.html
OK 20241002051244: ratings/20241002051244_tt7060344.html
Completed 12125/12789 movie/year tasks
Attempt failed: 20241002111816: chromedriver error

__TOR is OK
__TOR is OK
__TOR is OK
__TOR is OK
Attempt failed: 20221118211837: chromedriver error

Attempt failed: 20250620140159: chromedriver error

__TOR is OK
Attempt failed: 20230604020215: chromedriver error

Attempt failed: 20210417234359: chromedriver error

Attempt failed: 20221015092254: chromedriver error

Attempt failed: 20210524040822: chromedriver error

Attempt failed: 20210417234359: chromedriver error

Attempt failed: 20250620140159: chromedriver error

Attempt failed: 20220518194808: chromedriver error

OK

Metadata file /Users/surenpoghosyan/.cache/selenium/se-metadata.json cannot be parsed: EOF while parsing a value at line 18 column 26. Using empty metadata


OK 20150404171915: ratings/20150404171915_tt0099348.htmlAttempt failed: 20150404171915: chromedriver error

__TOR is OK
__TOR is OK
__TOR is OK
Attempt failed: 20220620215641: chromedriver error

Attempt failed: 20131112004428: chromedriver error

Attempt failed: 20241002042723: chromedriver error

Attempt failed: 20220925103618: chromedriver error

Attempt failed: 20080419065528: chromedriver error

Attempt failed: 20220208104028: chromedriver error

Attempt failed: 20210307163639: chromedriver error

OK 20230128170400: ratings/20230128170400_tt0099348.html
Attempt failed: 20051218184836: chromedriver error

Attempt failed: 20140705192855: chromedriver error

FAILED tt0099348 2017: tried 1 captures from 2017
__TOR is OK
__TOR is OK
OK 20230419085744: ratings/20230419085744_tt1877830.html
__TOR is OK
__TOR is OK
OK 20090912051009: ratings/20090912051009_tt0099348.html

SKIP 20150404171915: Already downloaded

SKIP 20161226181415: Already downloaded
FAILED tt0099348 2015: tried 1 captur

In [ ]:
# import csv
# from pathlib import Path

# output_file = Path("ratings_capture_dates.csv")
# fieldnames = ["imdb_id", "year", "timestamp", "date", "original"]

# with output_file.open("w", newline="", encoding="utf-8") as file:
#     writer = csv.DictWriter(file, fieldnames=fieldnames)
#     writer.writeheader()
#     writer.writerows(available_capture_dates)

# print(f"Saved {len(available_capture_dates)} capture records to {output_file.resolve()}")